In [38]:
import pandas as pd
import numpy as np

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings

warnings.filterwarnings('ignore')
sns.set_style("whitegrid")

In [39]:
fund_master = pd.read_csv("../data/processed/01_fund_master_cleaned.csv")

nav = pd.read_csv("../data/processed/02_nav_history_cleaned.csv")

aum = pd.read_csv("../data/processed/03_aum_by_fund_house_cleaned.csv")

sip = pd.read_csv("../data/processed/04_monthly_sip_inflows_cleaned.csv")

category = pd.read_csv("../data/processed/05_category_inflows_cleaned.csv")

folio = pd.read_csv("../data/processed/06_industry_folio_count_cleaned.csv")

performance = pd.read_csv("../data/processed/07_scheme_performance_cleaned.csv")

investor = pd.read_csv("../data/processed/08_investor_transactions_cleaned.csv")

holdings = pd.read_csv("../data/processed/09_portfolio_holdings_cleaned.csv")

benchmark = pd.read_csv("../data/processed/10_benchmark_indices_cleaned.csv")

In [40]:
print(nav.shape)
print(aum.shape)
print(sip.shape)

(64320, 3)
(90, 5)
(48, 6)


In [41]:
nav.isnull().sum()

amfi_code    0
date         0
nav          0
dtype: int64

In [42]:
nav["date"] = pd.to_datetime(nav["date"])
aum["date"] = pd.to_datetime(aum["date"])
benchmark["date"] = pd.to_datetime(benchmark["date"])

In [43]:
sip["month"] = pd.to_datetime(sip["month"], format="mixed")

In [44]:
category["month"] = pd.to_datetime(category["month"], format="mixed")

In [45]:
folio["month"] = pd.to_datetime(folio["month"], format="mixed")

In [46]:
import os

os.makedirs("../reports/charts", exist_ok=True)

In [47]:
nav.head()

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [48]:
nav = nav.merge(
    fund_master[["amfi_code", "scheme_name"]].drop_duplicates(),
    on="amfi_code",
    how="left"
)

In [49]:
nav["scheme_name"].nunique()

40

In [50]:
fig = px.line(
    nav,
    x="date",
    y="nav",
    color="scheme_name",
    title="NAV Trend 2022-2025"
)

fig.show()

In [51]:
fig.add_vrect(
    x0="2023-01-01",
    x1="2023-12-31",
    fillcolor="green",
    opacity=0.1,
    annotation_text="2023 Bull Run",
    annotation_position="top left"
)

fig.show()

In [52]:
fig.add_vrect(
    x0="2024-03-01",
    x1="2024-06-30",
    fillcolor="red",
    opacity=0.1,
    annotation_text="2024 Correction",
    annotation_position="top right"
)

fig.show()

In [53]:
fig.write_html("../reports/charts/nav_trend.html")
print("Saved: reports/charts/nav_trend.html")

Saved: reports/charts/nav_trend.html


In [54]:
aum["year"] = aum["date"].dt.year
aum.head()

,date,fund_house,aum_lakh_crore,aum_crore,num_schemes,year
0,2022-03-31,SBI Mutual Fund,6.05,605000,186,2022
1,2022-03-31,ICICI Prudential MF,4.65,465000,216,2022
2,2022-03-31,HDFC Mutual Fund,4.35,435000,195,2022
3,2022-03-31,Nippon India MF,2.70,270000,177,2022
4,2022-03-31,Kotak Mahindra MF,2.70,270000,168,2022


In [55]:
plt.figure(figsize=(15, 7))

sns.barplot(
    data=aum,
    x="year",
    y="aum_lakh_crore",
    hue="fund_house"
)

plt.show()

In [56]:
plt.figure(figsize=(15, 7))

sns.barplot(
    data=aum,
    x="year",
    y="aum_lakh_crore",
    hue="fund_house"
)

plt.title("Fund House AUM", fontsize=16, fontweight="bold")
plt.show()

In [57]:
plt.figure(figsize=(15, 7))

sns.barplot(
    data=aum,
    x="year",
    y="aum_lakh_crore",
    hue="fund_house"
)

plt.title("Fund House AUM", fontsize=16, fontweight="bold")
plt.xticks(rotation=45)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

In [58]:
plt.figure(figsize=(15, 7))

sns.barplot(
    data=aum,
    x="year",
    y="aum_lakh_crore",
    hue="fund_house"
)

plt.title("Fund House AUM", fontsize=16, fontweight="bold")
plt.xticks(rotation=45)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
plt.tight_layout()

plt.savefig("../reports/charts/aum_growth.png", dpi=150, bbox_inches="tight")
print("Saved: reports/charts/aum_growth.png")
plt.show()

Saved: reports/charts/aum_growth.png


In [59]:
fig = px.line(
    sip,
    x="month",
    y="sip_inflow_crore",
    title="Monthly SIP Inflows Trend",
    markers=True
)

fig.show()

In [60]:
sip.loc[
    sip["sip_inflow_crore"].idxmax()
]

month                        2025-12-01 00:00:00
sip_inflow_crore                           31002
active_sip_accounts_crore                   9.35
new_sip_accounts_lakh                        9.8
sip_aum_lakh_crore                          15.9
yoy_growth_pct                             17.17
Name: 47, dtype: object

In [61]:
peak_month = sip.loc[sip["sip_inflow_crore"].idxmax(), "month"]
peak_value = sip["sip_inflow_crore"].max()

fig.add_annotation(
    x=peak_month,
    y=peak_value,
    text=f"{peak_value:.0f} Cr",
    showarrow=True,
    arrowhead=2,
    bgcolor="red",
    font=dict(color="white", size=12)
)

fig.show()

In [62]:
fig.write_html(
    "../reports/charts/sip_trend.html"
)
print("Saved: reports/charts/sip_trend.html")

Saved: reports/charts/sip_trend.html


In [63]:
heatmap_data = category.pivot_table(
    index="category",
    columns="month",
    values="net_inflow_crore",
    aggfunc="sum"
)

heatmap_data.head()

month,2024-04-01,2024-05-01,2024-06-01,2024-07-01,2024-08-01,2024-09-01,2024-10-01,2024-11-01,2024-12-01,2025-01-01,2025-02-01,2025-03-01
category,,,,,,,,,,,,
ELSS,466.0,553.0,472.0,471.0,499.0,537.0,537.0,571.0,521.0,516.0,437.0,500.0
Flexi Cap,4947.0,5529.0,4478.0,4869.0,5562.0,5397.0,6004.0,6111.0,4654.0,5603.0,6068.0,4767.0
Gilt,784.0,836.0,864.0,959.0,952.0,925.0,898.0,704.0,831.0,744.0,942.0,956.0
Hybrid,2955.0,3487.0,3163.0,3291.0,3684.0,3015.0,3314.0,3264.0,3538.0,2967.0,3360.0,2830.0
Large & Mid Cap,4214.0,4368.0,4610.0,5023.0,5411.0,4528.0,4581.0,5556.0,4878.0,4816.0,5524.0,4243.0


In [64]:
plt.figure(figsize=(15, 8))

sns.heatmap(
    heatmap_data,
    cmap="YlGnBu",
    annot=True,
    fmt=".0f",
    linewidths=0.5
)

plt.show()

In [65]:
plt.figure(figsize=(15, 8))

sns.heatmap(
    heatmap_data,
    cmap="YlGnBu",
    annot=True,
    fmt=".0f",
    linewidths=0.5
)

plt.title("Category Inflow Heatmap", fontsize=16, fontweight="bold")
plt.xlabel("Month")
plt.ylabel("Category")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [66]:
plt.figure(figsize=(15, 8))

sns.heatmap(
    heatmap_data,
    cmap="YlGnBu",
    annot=True,
    fmt=".0f",
    linewidths=0.5
)

plt.title("Category Inflow Heatmap", fontsize=16, fontweight="bold")
plt.xlabel("Month")
plt.ylabel("Category")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

plt.savefig("../reports/charts/category_heatmap.png", dpi=150, bbox_inches="tight")
print("Saved: reports/charts/category_heatmap.png")
plt.show()

Saved: reports/charts/category_heatmap.png


In [67]:
investor["age_group"].value_counts()

age_group
26-35    13463
36-45     8146
18-25     4916
46-55     3779
56+       2474
Name: count, dtype: int64

In [68]:
plt.figure(figsize=(8, 8))

investor["age_group"].value_counts().plot.pie(
    autopct="%1.1f%%",
    startangle=140,
    colors=plt.cm.Set3.colors
)

plt.title("Investor Distribution by Age Group", fontsize=14, fontweight="bold")
plt.ylabel("")

plt.savefig("../reports/charts/age_distribution.png", dpi=150, bbox_inches="tight")
print("Saved: reports/charts/age_distribution.png")
plt.show()

Saved: reports/charts/age_distribution.png


In [69]:
investor["gender"].value_counts()

gender
Male      21809
Female    10969
Name: count, dtype: int64

In [70]:
plt.figure(figsize=(8, 5))

sns.countplot(
    data=investor,
    x="gender",
    palette=["#3498db", "#e74c3c", "#2ecc71"]
)

plt.title("Gender Split of Investors", fontsize=14, fontweight="bold")
plt.xlabel("Gender")
plt.ylabel("Count")

plt.savefig("../reports/charts/gender_split.png", dpi=150, bbox_inches="tight")
print("Saved: reports/charts/gender_split.png")
plt.show()

Saved: reports/charts/gender_split.png


In [71]:
sip_users = investor[
    investor["transaction_type"] == "SIP"
]

print(f"Total transactions: {len(investor)}")
print(f"SIP transactions:   {len(sip_users)}")

Total transactions: 32778
SIP transactions:   19716


In [72]:
plt.figure(figsize=(12, 6))

sns.boxplot(
    data=sip_users,
    x="age_group",
    y="amount_inr",
    palette="mako"
)

plt.title("SIP Amount by Age Group", fontsize=14, fontweight="bold")
plt.xlabel("Age Group")
plt.ylabel("Amount (INR)")

plt.savefig("../reports/charts/sip_boxplot.png", dpi=150, bbox_inches="tight")
print("Saved: reports/charts/sip_boxplot.png")
plt.show()

Saved: reports/charts/sip_boxplot.png


In [73]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


print("Columns in investor dataset:")
print(investor.columns.tolist())


state_sip = (
    investor
    .groupby("state")["amount_inr"]
    .sum()
    .sort_values(ascending=True)
)


print(state_sip.head())


plt.figure(figsize=(12, 8))

state_sip.plot.barh(
    color=plt.cm.magma(
        np.linspace(0.3, 0.8, len(state_sip))
    )
)

plt.title(
    "State-wise SIP Investment",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("Total SIP Amount (INR)")
plt.ylabel("State")

plt.tight_layout()


plt.savefig(
    "../reports/charts/state_distribution.png",
    dpi=150,
    bbox_inches="tight"
)

print("Saved: reports/charts/state_distribution.png")

plt.show()

Columns in investor dataset:
['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']
state
Maharashtra      269513480
Karnataka        273753570
Haryana          279634354
Uttar Pradesh    285368873
Delhi            289633404
Name: amount_inr, dtype: int64
Saved: reports/charts/state_distribution.png


In [74]:
investor["city_tier"].value_counts()

city_tier
T30    21719
B30    11059
Name: count, dtype: int64

In [75]:
plt.figure(figsize=(8, 8))

investor["city_tier"].value_counts().plot.pie(
    autopct="%1.1f%%",
    startangle=140,
    colors=["#e74c3c", "#f39c12", "#2ecc71", "#3498db"]
)

plt.title("Investment by City Tier", fontsize=14, fontweight="bold")
plt.ylabel("")

plt.savefig("../reports/charts/city_tier_distribution.png", dpi=150, bbox_inches="tight")
print("Saved: reports/charts/city_tier_distribution.png")
plt.show()

Saved: reports/charts/city_tier_distribution.png


In [76]:
plt.figure(figsize=(12, 5))

plt.plot(
    folio["month"],
    folio["total_folios_crore"],
    color="#2ecc71",
    linewidth=2.5
)

plt.show()

In [77]:
plt.figure(figsize=(12, 5))

plt.plot(
    folio["month"],
    folio["total_folios_crore"],
    color="#2ecc71",
    linewidth=2.5
)

plt.scatter(
    folio["month"],
    folio["total_folios_crore"],
    color="#27ae60",
    s=60,
    zorder=5,
    edgecolors="white"
)

plt.show()

In [78]:
plt.figure(figsize=(12, 5))

plt.plot(
    folio["month"],
    folio["total_folios_crore"],
    color="#2ecc71",
    linewidth=2.5,
    label="Total Folios"
)

plt.scatter(
    folio["month"],
    folio["total_folios_crore"],
    color="#27ae60",
    s=60,
    zorder=5,
    edgecolors="white"
)

plt.title(
    "Industry Folio Growth",
    fontsize=16,
    fontweight="bold"
)
plt.xlabel("Month")
plt.ylabel("Total Folios (Crores)")
plt.legend()
plt.tight_layout()
plt.show()

In [79]:
plt.figure(figsize=(12, 5))

plt.plot(
    folio["month"],
    folio["total_folios_crore"],
    color="#2ecc71",
    linewidth=2.5,
    label="Total Folios"
)

plt.scatter(
    folio["month"],
    folio["total_folios_crore"],
    color="#27ae60",
    s=60,
    zorder=5,
    edgecolors="white"
)

plt.title(
    "Industry Folio Growth",
    fontsize=16,
    fontweight="bold"
)
plt.xlabel("Month")
plt.ylabel("Total Folios (Crores)")
plt.legend()
plt.tight_layout()

plt.savefig(
    "../reports/charts/folio_growth.png",
    dpi=150,
    bbox_inches="tight"
)
print("Saved: reports/charts/folio_growth.png")
plt.show()

Saved: reports/charts/folio_growth.png


In [80]:
top10 = nav["amfi_code"].unique()[:10]
print(f"Selected {len(top10)} funds")
print(top10)

Selected 10 funds
[119551 119552 119598 119599 119120 100016 125497 100033 125498 100025]


In [81]:
corr_df = nav[
    nav["amfi_code"].isin(top10)
]

print(f"Filtered shape: {corr_df.shape}")

Filtered shape: (16080, 4)


In [82]:
pivot = corr_df.pivot_table(
    index="date",
    columns="amfi_code",
    values="nav",
    aggfunc="first"
)

print(f"Pivot shape: {pivot.shape}")
pivot.head()

Pivot shape: (1608, 10)


amfi_code,100016,100025,100033,119120,119551,119552,119598,119599,125497,125498
date,,,,,,,,,,
2022-01-03,520.4608,26.3169,107.3758,42.1391,54.3856,58.4174,89.8738,96.4565,560.1443,117.5969
2022-01-04,515.0971,26.2234,105.9447,42.2508,54.3474,57.3480,88.5495,94.6512,560.7052,117.0077
2022-01-05,521.7239,26.2221,105.4800,42.4374,54.6869,57.0552,88.0925,94.5436,563.0884,116.4011
2022-01-06,515.7880,26.1728,104.9350,42.5901,55.4550,56.4224,88.5175,93.7944,561.0675,116.0861
2022-01-07,515.1639,26.2261,104.3318,42.4851,55.3692,57.2750,91.4235,89.6438,559.5420,114.6164


In [83]:
returns = pivot.pct_change().dropna()

print(f"Returns shape: {returns.shape}")
returns.head()

Returns shape: (1607, 10)


amfi_code,100016,100025,100033,119120,119551,119552,119598,119599,125497,125498
date,,,,,,,,,,
2022-01-04,-0.010306,-0.003553,-0.013328,0.002651,-0.000702,-0.018306,-0.014735,-0.018716,0.001001,-0.005010
2022-01-05,0.012865,-0.000050,-0.004386,0.004416,0.006247,-0.005106,-0.005161,-0.001137,0.004250,-0.005184
2022-01-06,-0.011377,-0.001880,-0.005167,0.003598,0.014045,-0.011091,0.004824,-0.007924,-0.003589,-0.002706
2022-01-07,-0.001210,0.002036,-0.005748,-0.002465,-0.001547,0.015111,0.032830,-0.044252,-0.002719,-0.012660
2022-01-08,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [84]:
corr = returns.corr()

corr

amfi_code,100016,100025,100033,119120,119551,119552,119598,119599,125497,125498
amfi_code,,,,,,,,,,
100016,1.000000,0.045839,0.000393,-0.022104,0.041257,0.014040,-0.029936,0.000049,0.047436,-0.015903
100025,0.045839,1.000000,0.003928,-0.039060,0.019062,-0.001037,-0.063428,-0.033784,0.012820,-0.024669
100033,0.000393,0.003928,1.000000,-0.004859,-0.012789,-0.025104,0.008600,-0.027959,-0.023108,-0.010835
119120,-0.022104,-0.039060,-0.004859,1.000000,0.024222,-0.001730,0.020308,0.018099,-0.008577,-0.015248
119551,0.041257,0.019062,-0.012789,0.024222,1.000000,-0.004106,0.026635,-0.072056,0.018743,0.031642
119552,0.014040,-0.001037,-0.025104,-0.001730,-0.004106,1.000000,0.022644,-0.042040,0.029696,-0.051176
119598,-0.029936,-0.063428,0.008600,0.020308,0.026635,0.022644,1.000000,0.018448,-0.051531,0.019315
119599,0.000049,-0.033784,-0.027959,0.018099,-0.072056,-0.042040,0.018448,1.000000,0.003624,0.042066
125497,0.047436,0.012820,-0.023108,-0.008577,0.018743,0.029696,-0.051531,0.003624,1.000000,0.028267


In [85]:
plt.figure(figsize=(10, 8))

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    square=True
)

plt.title("NAV Return Correlation Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [86]:
plt.figure(figsize=(10, 8))

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    square=True
)

plt.title("NAV Return Correlation Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()

plt.savefig(
    "../reports/charts/correlation_matrix.png",
    dpi=150,
    bbox_inches="tight"
)
print("Saved: reports/charts/correlation_matrix.png")
plt.show()

Saved: reports/charts/correlation_matrix.png


In [87]:
sector = (
    holdings
    .groupby("sector")["weight_pct"]
    .sum()
)

sector.sort_values(ascending=False)

sector
Banking           652.26
IT                455.47
Pharma            407.45
Automobile        323.65
Utilities         265.54
FMCG              229.11
Infrastructure    192.16
Diversified       169.23
Telecom           145.62
Consumer Goods    127.61
NBFC              119.09
Energy            117.91
Cement            105.03
Paints             89.86
Name: weight_pct, dtype: float64

In [88]:
fig = go.Figure(
    go.Pie(
        labels=sector.index,
        values=sector.values,
        hole=0.5,
        textinfo="label+percent",
        textposition="outside"
    )
)

fig.update_layout(
    title="Sector Allocation (Donut Chart)",
    height=600
)

fig.show()

In [89]:
fig.write_html(
    "../reports/charts/sector_donut.html"
)
print("Saved: reports/charts/sector_donut.html")

Saved: reports/charts/sector_donut.html
